# Orkiestracja Strojenia Hiperparametrów (HyperTuning)

Niniejszy notatnik automatyzuje proces poszukiwania najlepszych hiperparametrów algorytmem TPE (Tree of Parzen Estimators) dla zestawu wybranych modeli. Modele przeszukują zdefiniowane przestrzenie parametrów (`*.hyper`) i zapisują raporty końcowe do folderu `3_Evaluation/Reports/`.

**Uwaga:** Uruchomienie całego notatnika (*Run All*) może zająć od kilkunastu minut do kilkunastu godzin w zależności od mocy obliczeniowej karty graficznej i liczby iteracji.

In [ ]:
import os
import sys

# Przejście do głównego katalogu projektu, aby poprawnie resolwować ścieżki
if os.path.basename(os.getcwd()) == 'Notebooks':
    os.chdir('..')
    
print("Aktualny katalog roboczy:", os.getcwd())

### 1. Optymalizacja ItemKNN

In [ ]:
!{sys.executable} 2_Experiments/run_hyper_itemknn.py

### 2. Optymalizacja BPR-MF

In [ ]:
!{sys.executable} 2_Experiments/run_hyper_bpr.py

### 3. Optymalizacja NeuMF (NCF)

In [ ]:
!{sys.executable} 2_Experiments/run_hyper_ncf.py

### 4. Optymalizacja LightGCN (GNN)

In [ ]:
!{sys.executable} 2_Experiments/run_hyper_gnn.py

## Zestawienie Wyników Ostatecznych

Poniższy kod iteruje przez wszystkie pliki `*.result` wygenerowane przez RecBole w folderze `3_Evaluation/Reports/`. Analizuje je by wyłuskać najlepsze wyniki z fazy testowej, po czym sortuje od największego NDCG@10 do najgorszego.

In [ ]:
import glob
import pandas as pd
import ast

def extract_metrics_from_result_file(file_path):
    """
    Funkcja parsująca zrzut słownika wygenerowany przez RecBole w pliku .result
    """
    with open(file_path, 'r', encoding='utf-8') as f:
        content = f.read()
        
    try:
        # RecBole zapisuje wynik używając formatu słownika pythonowego
        # np. {'best_valid_score': 0.12, 'best_valid_result': {...}, 'test_result': {...}}
        # Należy znaleźć linię rozpoczynającą się od { albo spróbować evalu
        # Dla prostoty wyodrębnimy słownik przy użyciu biblioteki ast.
        # Często raport jest po prostu wypisaniem dicte'a, więc szukamy pasujących nawiasów
        dict_start = content.find('{')
        dict_end = content.rfind('}') + 1
        if dict_start != -1 and dict_end != -1:
            dict_str = content[dict_start:dict_end]
            data = ast.literal_eval(dict_str)
            
            # Pobieramy wyniki testowe 
            test_result = data.get('test_result', {})
            best_params = data.get('best_params', {})
            
            # Zabezpieczenie nazw kluczy (NDCG@10 lub ndcg@10)
            test_metrics = {k.upper(): v for k, v in test_result.items()}
            
            import re
            time_match = re.search(r"'Optimization_Time_Seconds':\s*([0-9.]+)", content)
            opt_time = round(float(time_match.group(1)) / 60, 2) if time_match else 0.0

            return {
                'Model': file_path.split('_')[-2].upper(),  # heurystyka do wyciągnięcia nazwy np. bpr z hyper_results_bpr_ml-100k.result
                'NDCG@10': test_metrics.get('NDCG@10', 0.0),
                'Recall@10': test_metrics.get('RECALL@10', 0.0),
                'Hit@10': test_metrics.get('HIT@10', 0.0),
                'Czas_optymalizacji [min]': opt_time,
                'Best_Params': str(best_params)
            }
    except Exception as e:
        print(f"Nie udało się sparsować {file_path}: {e}")
    
    return None

results = []
report_files = glob.glob('3_Evaluation/Reports/hyper_results_*.result')

for path in report_files:
    parsed = extract_metrics_from_result_file(path)
    if parsed:
        results.append(parsed)

if results:
    df = pd.DataFrame(results)
    # Sortowanie od najlepszego NDCG@10
    df = df.sort_values(by='NDCG@10', ascending=False).reset_index(drop=True)
    
    print("\n=== RANKING OPTYMALIZACJI MODELI (ZBIÓR TESTOWY) ===\n")
    display(df)
else:
    print("Nie znaleziono poprawnych wyników w folderze 3_Evaluation/Reports/. Upewnij się, że proces HyperTuningu się zakończył.")
